# Entity extraction results analysis
This is a demo for the exploration of results yielded by entity extraction on a representative subset of GoTriple data. 


### Imports

In [1]:
import pandas as pd
from pathlib import Path
from itertools import combinations
import asyncio
import aiohttp
import json


### Constants

In [2]:
DATA_PATH = "./merged/" # This folder should contain all the json gathering data for each discipline
COMPARABLE_MODELS = ("spacy", "gliner", "base-llm")
CONVERSION_TABLE = {
    "person name": "PERS",
    "organization": "ORG",
    "location name": "LOC",
    "time references": "MISC"
}
LINKER_URL = "https://rel.cs.ru.nl/api/"
MAX_CONC = 5

### Data preparation

In [3]:
def reshape_jsons(dir):
    for f in Path(dir).iterdir():
        if not f.name.endswith("json"):
            continue
        with open(f, mode="r", encoding="utf-8") as i:
            data = json.load(i)
        with open(f, mode="w", encoding="utf-8") as o:
            json.dump(data, o, indent=2, ensure_ascii=False)
            

df = pd.concat(pd.read_json(f, orient="records") for f in Path(DATA_PATH).iterdir())
print(df)
df = df.explode(column="ents")
df["chunk_idx"] = df["ents"].apply(lambda r: r[0])
df["ents"] = df["ents"].apply(lambda r: r[1])
df = df.explode(column="ents")

df = df.join(pd.json_normalize(df["ents"]))
df = df.drop(columns=["ents", "start", "end", "score"])
print(df)

      doc_id     model       lang  \
0     demo_0     spacy         en   
1     demo_0    gliner         en   
2     demo_1     spacy         en   
3     demo_1    gliner         en   
4     demo_2     spacy         en   
..       ...       ...        ...   
193  info_61  base-llm  undefined   
194  info_62  base-llm  undefined   
195  info_63  base-llm  undefined   
196  info_64  base-llm      other   
197  info_65  base-llm      other   

                                                  ents  
0    [[0, [{'start': 0, 'end': 203, 'text': 'Univer...  
1    [[0, [{'start': 0, 'end': 27, 'text': 'Univers...  
2    [[0, [{'start': 100, 'end': 109, 'text': 'This...  
3    [[0, [{'start': 160, 'end': 173, 'text': 'Walt...  
4    [[0, [{'start': 0, 'end': 94, 'text': 'Univers...  
..                                                 ...  
193  [[0, [{'text': 'Kean University', 'label': 'or...  
194  [[0, [{'text': 'University of Northern Iowa', ...  
195  [[0, []], [1, [{'text': 'Collections 

## Entity Label Distributions

In [4]:
print(df.value_counts("label"))

label
MISC               350167
PER                272324
ORG                196466
organization       179993
time references    167923
LOC                157886
location name       86570
person name         77083
Name: count, dtype: int64


    TAKEAWAY
Spacy extracts way more labels than the LLM, probably due to: 

* high uppercase sensitivity

* MISC being a very broad label

## Model Agreement

In [6]:
def make_clean_set(series):
    # drop NaN and None
    cleaned = {x for x in series if pd.notna(x)}
    return cleaned if cleaned else set()

# Getting enttity sets for each chunk in each document
model_sets = (
    df.groupby(["doc_id", "chunk_idx", "model"])["text"]
      .agg(make_clean_set)
      .reset_index()
)

model_sets.rename(columns={"text": "entity_set"}, inplace=True)

In [7]:
# Pivoting to compare the models
pivot_df = model_sets.pivot(
    index=["doc_id", "chunk_idx"],
    columns="model",
    values="entity_set"
)
print(pivot_df)


model                                   base-llm  \
doc_id       chunk_idx                             
anthro-bio_0 0          {Digital Commons @ Univ}   
             1          {Digital Commons @ Univ}   
             2          {Digital Commons @ Univ}   
             3          {Digital Commons @ Univ}   
             4          {Digital Commons @ Univ}   
...                                          ...   
stat_9       27                   {1862 July 27}   
             28                   {1862 July 27}   
             29                   {1862 July 27}   
             30                   {1862 July 27}   
             31                   {1862 July 27}   

model                                                              gliner  \
doc_id       chunk_idx                                                      
anthro-bio_0 0                                                    {Moore}   
             1                                                    {Moore}   
             2 

In [ ]:
def compute_jaccard(comp_df):
    comp_df["intersection"] = comp_df.apply(
        lambda r: len(r[model_a] & r[model_b]), axis=1
    ) # getting intersection between entity set predictions
    comp_df["union"] = comp_df.apply(
        lambda r: len(r[model_a] | r[model_b] ), axis=1
    ) # getting the union of the entity sets
    comp_df["jaccard"] = comp_df["intersection"] / comp_df["union"]
    comp_df[f"{model_a}_size"] = comp_df[model_a].apply(len)
    comp_df[f"{model_b}_size"] = comp_df[model_b].apply(len)

    return comp_df

# Compute Jaccard score between the models (chunk-wise)
for model_a, model_b in combinations(COMPARABLE_MODELS, r=2):
    print("-" * 50)
    print(f"COMPARING {model_a} and {model_b}")
    print("-" * 50)
    comp_df = pivot_df.copy()
    final_chunkwise = compute_jaccard(comp_df=comp_df)
    print("*** CHUNK-WISE STATS ***")
    print("\n")
    print(f"Mean model agreement: \n{final_chunkwise["jaccard"].mean()}\n")
    print(f"Aggrement stats: \n{final_chunkwise["jaccard"].describe()}\n")
    print(f"Value counts: \n{final_chunkwise["jaccard"].value_counts()}")

    # Combining each chunk entity set into a document set of entities 
    doc_grouped = comp_df.groupby("doc_id").agg(
        {
            model_a: lambda sets: set().union(*sets),
            model_b: lambda sets: set().union(*sets),
        }
    )
    
    final_docwise = compute_jaccard(doc_grouped)
    print("\n\n\n")
    print("*** DOC-WISE STATS ***")
    print("\n")
    print(f"Mean model agreement: \n{final_docwise["jaccard"].mean()}\n")
    print(f"Agreement stats: \n{final_docwise["jaccard"].describe()}\n")
    print(f"Value counts: \n{final_docwise["jaccard"].value_counts()}")



--------------------------------------------------
COMPARING spacy and gliner
--------------------------------------------------
*** CHUNK-WISE STATS ***


Mean model agreement: 
0.06150175961017867

Aggrement stats: 
count    118208.000000
mean          0.061502
std           0.240249
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
Name: jaccard, dtype: float64

Value counts: 
jaccard
0.0    110938
1.0      7270
Name: count, dtype: int64




*** DOC-WISE STATS ***


Mean model agreement: 
0.0818926296633303

Aggrement stats: 
count    2198.000000
mean        0.081893
std         0.274263
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000
Name: jaccard, dtype: float64

Value counts: 
jaccard
0.0    2018
1.0     180
Name: count, dtype: int64
--------------------------------------------------
COMPARING spacy and base-llm
------------------------------------------------